# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [1]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

Cloning into 'SQuAD-explorer'...
remote: Enumerating objects: 5563, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 5563 (delta 11), reused 17 (delta 6), pack-reused 5539 (from 1)
Receiving objects: 100% (5563/5563), 52.26 MiB | 60.33 MiB/s, done.
Resolving deltas: 100% (3563/3563), done.


In [2]:
import json

In [3]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [4]:
type(train_data)

dict

In [5]:
list(train_data.keys())

['data', 'version']

In [6]:
type(train_data["data"])

list

In [7]:
len(train_data["data"])

442

In [8]:
type(train_data["data"][0])

dict

In [9]:
train_data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [10]:
train_data["data"][0]["title"]

'University_of_Notre_Dame'

In [11]:
len(train_data["data"][0]["paragraphs"])

55

In [12]:
train_data["data"][0]["paragraphs"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [13]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [14]:
import pandas as pd

rows = []
for doc in train_data["data"]:
    title = doc["title"]
    for idx, para in enumerate(doc["paragraphs"]):
        rows.append({
            "document_title": title,
            "paragraph_index": idx,
            "paragraph_context": para["context"]
        })

parsed_df = pd.DataFrame(rows)
parsed_df.to_csv("parsed.tsv", sep="\t", index=False)
print(f"Saved parsed.tsv with {len(parsed_df)} rows")
parsed_df.head(3)

Saved parsed.tsv with 18896 rows


,document_title,paragraph_index,paragraph_context
0,University_of_Notre_Dame,0,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,1,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,2,The university is the major seat of the Congre...


Submit "parsed.tsv" in Gradescope.

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [15]:
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF with 1024 features on all paragraph contexts
vectorizer = TfidfVectorizer(max_features=1024, sublinear_tf=True, stop_words="english")
paragraph_matrix = vectorizer.fit_transform(parsed_df["paragraph_context"])
paragraph_vectors = paragraph_matrix.toarray()
print(f"Paragraph vectors shape: {paragraph_vectors.shape}")

Paragraph vectors shape: (18896, 1024)


Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [16]:
para_vec_df = parsed_df[["document_title", "paragraph_index"]].copy()
para_vec_df["paragraph_vector_json"] = [json.dumps(v.tolist()) for v in paragraph_vectors]
para_vec_df.to_csv("paragraph-vectors.tsv.gz", sep="\t", index=False, compression="gzip")
print(f"Saved paragraph-vectors.tsv.gz with {len(para_vec_df)} rows, vector length 1024")

Saved paragraph-vectors.tsv.gz with 18896 rows, vector length 1024


Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [17]:
questions_df = pd.read_csv("questions.tsv", sep="\t")
# Encode questions with the same fitted vectorizer
question_matrix = vectorizer.transform(questions_df["question"])
question_vectors = question_matrix.toarray()
print(f"Question vectors shape: {question_vectors.shape}")

Question vectors shape: (100, 1024)


Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [18]:
q_vec_df = questions_df[["question_id"]].copy()
q_vec_df["question_vector_json"] = [json.dumps(v.tolist()) for v in question_vectors]
q_vec_df.to_csv("question-vectors.tsv", sep="\t", index=False)
print(f"Saved question-vectors.tsv with {len(q_vec_df)} rows, vector length 1024")

Saved question-vectors.tsv with 100 rows, vector length 1024


Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [19]:
from scipy.spatial.distance import cdist

# Compute pairwise Euclidean distances: (n_questions, n_paragraphs)
distances = cdist(question_vectors, paragraph_vectors, metric="euclidean")

# For each question, collect top 5 nearest paragraphs
matches = []
for q_idx in range(len(questions_df)):
    q_id = questions_df.iloc[q_idx]["question_id"]
    top5_indices = np.argsort(distances[q_idx])[:5]
    for rank, p_idx in enumerate(top5_indices, start=1):
        matches.append({
            "question_id": q_id,
            "question_rank": rank,
            "document_title": parsed_df.iloc[p_idx]["document_title"],
            "paragraph_index": int(parsed_df.iloc[p_idx]["paragraph_index"])
        })

matches_df = pd.DataFrame(matches)
matches_df.head(10)

,question_id,question_rank,document_title,paragraph_index
0,1,1,Genome,8
1,1,2,Plymouth,74
2,1,3,Near_East,57
3,1,4,Macintosh,21
4,1,5,Tuvalu,49
5,4,1,"Atlantic_City,_New_Jersey",12
6,4,2,Beyoncé,12
7,4,3,Genocide,12
8,4,4,Royal_Dutch_Shell,23
9,4,5,Dwight_D._Eisenhower,49


Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [20]:
matches_df.to_csv("question-matches.tsv", sep="\t", index=False)
print(f"Saved question-matches.tsv with {len(matches_df)} rows")

Saved question-matches.tsv with 500 rows


Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


Submit "worst-paragraphs.tsv" in Gradescope.

Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

In [21]:
# First 5 question_ids sorted ascending
first5_qids = sorted(questions_df["question_id"].tolist())[:5]

# Worst match = rank 5 (farthest neighbor among top 5)
worst_df = (
    matches_df[
        matches_df["question_id"].isin(first5_qids) &
        (matches_df["question_rank"] == 5)
    ][["question_id", "document_title", "paragraph_index"]]
    .sort_values("question_id")
    .reset_index(drop=True)
)

worst_df.to_csv("worst-paragraphs.tsv", sep="\t", index=False)
print("Saved worst-paragraphs.tsv")
worst_df

Saved worst-paragraphs.tsv


,question_id,document_title,paragraph_index
0,1,Tuvalu,49
1,4,Dwight_D._Eisenhower,49
2,7,Translation,22
3,10,Josip_Broz_Tito,52
4,13,Alaska,8


## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.